# Step 1 — Safety Fine-Tuning & Subspace Extraction

## What this step does
Fine-tunes `Qwen/Qwen2.5-7B` on HH-RLHF *chosen* (safe) responses using LoRA (r=16, α=32).
After training, extracts per-layer safety subspaces via SVD of the LoRA weight deltas and saves
them as projection matrices to disk.

## Requirements from previous steps
None — this is the first step.

## Produces
- `./checkpoints/safety_lora/` — LoRA adapter weights from safety fine-tuning
- `./checkpoints/safety_subspace.pt` — dict mapping layer names → projection matrices (U tensors)


In [9]:
# Cell 2: GPU check + environment info
import subprocess
import sys
import os

print("=" * 60)
print("ENVIRONMENT INFO")
print("=" * 60)
print(f"Python: {sys.version}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        n_gpus = torch.cuda.device_count()
        print(f"Number of GPUs: {n_gpus}")
        for i in range(n_gpus):
            props = torch.cuda.get_device_properties(i)
            mem_gb = props.total_memory / 1024**3
            print(f"  GPU {i}: {props.name} ({mem_gb:.1f} GB)")
except ImportError:
    print("PyTorch not yet installed — will be installed in next cell.")

print("=" * 60)

ENVIRONMENT INFO
Python: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
PyTorch: 2.8.0+cu128
CUDA available: True
CUDA version: 12.8
Number of GPUs: 2
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition (95.0 GB)
  GPU 1: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition (95.0 GB)


In [10]:
%%bash
pip install -q transformers datasets peft flwr evaluate

In [11]:
# Cell 4: All imports + project config

import os
import sys
import copy
import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)
from datasets import load_dataset, Dataset as HFDataset
import evaluate
import flwr as fl
from flwr.common import (
    NDArrays,
    Scalar,
    ndarrays_to_parameters,
    parameters_to_ndarrays,
    FitIns,
    FitRes,
    EvaluateIns,
    EvaluateRes,
    GetParametersIns,
    GetParametersRes,
    Parameters,
    Status,
    Code,
)
from flwr.server.strategy import FedAvg
from flwr.server.client_manager import ClientManager
from flwr.server.client_proxy import ClientProxy

warnings.filterwarnings("ignore")

# ── Project Configuration ──────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-7B"

LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

SAFETY_RANK = 8          # number of top singular vectors for safety subspace
N_CLIENTS = 10           # federated clients
N_ROUNDS = 3             # FL rounds
N_SAFETY_SAMPLES = 1000  # chosen responses for safety fine-tuning
N_BIASED_SAMPLES = 1000  # rejected responses for biased clients
N_DPO_PAIRS = 500        # preference pairs for DPO
MAX_SEQ_LEN = 512
BATCH_SIZE = 2
GRAD_ACCUM = 8
DTYPE = torch.bfloat16
BETA_DPO = 0.1

BASE_DIR = Path.home() / "federated-ethical-llm"
MODELS_DIR = BASE_DIR / "models"
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"

for d in [MODELS_DIR, DATA_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAFETY_SUBSPACE_PATH = MODELS_DIR / "safety_subspace.pt"
SAFETY_LORA_PATH = MODELS_DIR / "safety_lora"
FEDAVG_LORA_PATH = MODELS_DIR / "fedavg_lora.pt"
DPO_LORA_PATH = MODELS_DIR / "dpo_lora"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project configuration loaded.")
print(f"Base directory : {BASE_DIR}")
print(f"Primary device : {DEVICE}")
print(f"Model          : {MODEL_NAME}")
print(f"LoRA r/alpha   : {LORA_CONFIG.r}/{LORA_CONFIG.lora_alpha}")
print(f"Safety rank    : {SAFETY_RANK}")

Project configuration loaded.
Base directory : /home/jovyan/federated-ethical-llm
Primary device : cuda
Model          : Qwen/Qwen2.5-7B
LoRA r/alpha   : 16/32
Safety rank    : 8


---
## Step 1 — Safety Fine-Tuning & Subspace Extraction

1. Load `Qwen/Qwen2.5-7B` with LoRA (r=16, α=32).
2. Fine-tune on HH-RLHF **chosen** (safe) responses.
3. For each LoRA layer compute `delta_W = lora_B @ lora_A`, run SVD, and keep the top-k left singular vectors as the **safety basis** for that layer.
4. Save both the LoRA checkpoint and the safety subspace dict to disk.

Each subsequent step can reload from these checkpoints independently.


In [12]:
# Cell 6: Load model + tokenizer

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # synchronous errors for better tracebacks

def load_base_model_and_tokenizer(model_name: str = MODEL_NAME, for_training: bool = True):
    """Load Qwen2.5-7B. Use single GPU during training to avoid device_map conflicts."""
    print(f"Loading tokenizer for {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side="right",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Force single GPU for training — 7B model is ~14GB, well within 95GB
    device_map = {"": 0} if for_training else "auto"
    print(f"Loading model (bfloat16, device_map={device_map}) ...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=DTYPE,
        device_map=device_map,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model.enable_input_require_grads()
    print("Model loaded successfully.")
    return model, tokenizer


def apply_lora(model, lora_config: LoraConfig = LORA_CONFIG):
    """Wrap model with PEFT LoRA."""
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


print("Model loading helpers defined.")

Model loading helpers defined.


In [13]:
# Cell 7: Load + preprocess HH-RLHF

def load_hh_rlhf(n_chosen: int = N_SAFETY_SAMPLES,
                 n_rejected: int = N_BIASED_SAMPLES,
                 seed: int = 42):
    print("Loading HH-RLHF dataset ...")
    ds = load_dataset("Anthropic/hh-rlhf", split="train")
    ds = ds.shuffle(seed=seed)

    chosen_texts: List[str] = []
    rejected_texts: List[str] = []

    for example in ds:
        if len(chosen_texts) < n_chosen:
            chosen_texts.append(example["chosen"])
        if len(rejected_texts) < n_rejected:
            rejected_texts.append(example["rejected"])
        if len(chosen_texts) >= n_chosen and len(rejected_texts) >= n_rejected:
            break

    print(f"Collected {len(chosen_texts)} chosen and {len(rejected_texts)} rejected samples.")
    return chosen_texts, rejected_texts


class TextDataset(Dataset):
    """Simple tokenized text dataset with proper label masking."""

    def __init__(self, texts: List[str], tokenizer, max_length: int = MAX_SEQ_LEN):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        input_ids = self.encodings["input_ids"][idx]
        attention_mask = self.encodings["attention_mask"][idx]
        labels = input_ids.clone()
        # Mask padding tokens so they don't contribute to loss
        labels[attention_mask == 0] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


print("Dataset helpers defined.")

Dataset helpers defined.


In [14]:
# Cell 8: SafetyTrainer class

from transformers import default_data_collator

class SafetyTrainer:
    """Fine-tunes a LoRA-wrapped model on safe (chosen) responses."""

    def __init__(
        self,
        model,
        tokenizer,
        output_dir: str = str(SAFETY_LORA_PATH),
        num_train_epochs: int = 1,
        per_device_train_batch_size: int = BATCH_SIZE,
        gradient_accumulation_steps: int = GRAD_ACCUM,
        learning_rate: float = 2e-4,
        bf16: bool = True,
        logging_steps: int = 10,
        save_steps: int = 200,
        save_total_limit: int = 1,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=per_device_train_batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            learning_rate=learning_rate,
            fp16=False,
            bf16=bf16,
            logging_steps=logging_steps,
            save_steps=save_steps,
            save_total_limit=save_total_limit,
            remove_unused_columns=False,
            report_to="none",
            dataloader_num_workers=0,
        )

    def fine_tune(self, train_texts: List[str]) -> None:
        print(f"Building dataset with {len(train_texts)} samples ...")
        train_dataset = TextDataset(train_texts, self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=self.training_args,
            train_dataset=train_dataset,
            data_collator=default_data_collator,  # preserves our -100 labels
        )

        print("Starting safety fine-tuning ...")
        trainer.train()
        print(f"Fine-tuning complete. Saving to {self.training_args.output_dir}")
        self.model.save_pretrained(self.training_args.output_dir)
        self.tokenizer.save_pretrained(self.training_args.output_dir)


print("SafetyTrainer defined.")

SafetyTrainer defined.


In [15]:
# Cell 9: SafetySubspace class — SVD-based subspace extraction

class SafetySubspace:
    """
    Computes and stores the per-layer safety subspace from a LoRA-fine-tuned model.

    For each LoRA layer:
        delta_W = lora_B @ lora_A   (shape: out_features x in_features)
        U, S, Vt = svd(delta_W, full_matrices=False)
        safety_basis[layer_name] = U[:, :safety_rank]   # top-k left singular vectors
    """

    def __init__(self, safety_rank: int = SAFETY_RANK):
        self.safety_rank = safety_rank
        self.bases: Dict[str, torch.Tensor] = {}

    def compute_from_lora(self, peft_model) -> Dict[str, torch.Tensor]:
        """
        Extract safety basis matrices from all LoRA layers in the PEFT model.
        Returns a dict mapping layer_name → U tensor (CPU, float32).
        """
        print("Computing safety subspace via SVD of LoRA weight deltas ...")
        self.bases = {}

        named_modules = dict(peft_model.named_modules())

        # Iterate over lora_B parameters; pair with corresponding lora_A
        for name, param in peft_model.named_parameters():
            if "lora_B" not in name:
                continue

            # Derive lora_A name
            lora_a_name = name.replace("lora_B", "lora_A")
            lora_a_param = dict(peft_model.named_parameters()).get(lora_a_name)
            if lora_a_param is None:
                continue

            with torch.no_grad():
                lora_B = param.data.float()        # (out_features, r)
                lora_A = lora_a_param.data.float() # (r, in_features)
                delta_W = lora_B @ lora_A          # (out_features, in_features)

                try:
                    U, S, Vt = torch.linalg.svd(delta_W, full_matrices=False)
                    k = min(self.safety_rank, U.shape[1])
                    # layer_name is everything before ".lora_B"
                    layer_name = name.replace(".lora_B.default", "").replace(".lora_B", "")
                    self.bases[layer_name] = U[:, :k].cpu()  # store on CPU
                except Exception as e:
                    print(f"  SVD failed for {name}: {e}")
                    continue

        print(f"Safety subspace computed for {len(self.bases)} layers.")
        return self.bases

    def save(self, path: Path = SAFETY_SUBSPACE_PATH) -> None:
        """Persist the safety bases dict to disk."""
        torch.save(self.bases, str(path))
        print(f"Safety subspace saved to {path}")

    @classmethod
    def load(cls, path: Path = SAFETY_SUBSPACE_PATH, safety_rank: int = SAFETY_RANK):
        """Load a previously saved SafetySubspace."""
        obj = cls(safety_rank=safety_rank)
        obj.bases = torch.load(str(path), map_location="cpu")
        print(f"Safety subspace loaded from {path} ({len(obj.bases)} layers)")
        return obj


print("SafetySubspace defined.")

SafetySubspace defined.


In [16]:
# Cell 10: Run Step 1 — fine-tune + compute + save subspace
# This cell is independently runnable; it saves checkpoints to disk.

STEP1_DONE_FLAG = MODELS_DIR / "step1_done.flag"

if STEP1_DONE_FLAG.exists():
    print("Step 1 checkpoint found — skipping fine-tuning.")
    print(f"  Safety LoRA  : {SAFETY_LORA_PATH}")
    print(f"  Safety subspace: {SAFETY_SUBSPACE_PATH}")
else:
    # 1a. Load base model and tokenizer
    base_model, tokenizer = load_base_model_and_tokenizer()

    # 1b. Wrap with LoRA
    lora_model = apply_lora(base_model)

    # 1c. Load HH-RLHF data
    chosen_texts, rejected_texts = load_hh_rlhf(
        n_chosen=N_SAFETY_SAMPLES,
        n_rejected=N_BIASED_SAMPLES,
    )
    # Save rejected texts for use in Step 2
    torch.save(rejected_texts, str(DATA_DIR / "rejected_texts.pt"))
    # Save a sample of preference pairs for DPO (Step 3)
    dpo_pairs = list(zip(
        chosen_texts[:N_DPO_PAIRS],
        rejected_texts[:N_DPO_PAIRS],
    ))
    torch.save(dpo_pairs, str(DATA_DIR / "dpo_pairs.pt"))

    # 1d. Safety fine-tune
    trainer = SafetyTrainer(lora_model, tokenizer)
    trainer.fine_tune(chosen_texts)

    # 1e. Compute safety subspace via SVD
    safety_subspace = SafetySubspace(safety_rank=SAFETY_RANK)
    safety_subspace.compute_from_lora(lora_model)
    safety_subspace.save(SAFETY_SUBSPACE_PATH)

    # Mark step done
    STEP1_DONE_FLAG.touch()
    print("Step 1 complete.")

    # Free memory before next step
    del base_model, lora_model
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

Loading tokenizer for Qwen/Qwen2.5-7B ...
Loading model (bfloat16, device_map={'': 0}) ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully.
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
Loading HH-RLHF dataset ...
Collected 1000 chosen and 1000 rejected samples.
Building dataset with 1000 samples ...
Starting safety fine-tuning ...


Step,Training Loss
10,0.938200
20,0.830400
30,0.941600


Fine-tuning complete. Saving to /home/jovyan/federated-ethical-llm/models/safety_lora
Computing safety subspace via SVD of LoRA weight deltas ...
Safety subspace computed for 196 layers.
Safety subspace saved to /home/jovyan/federated-ethical-llm/models/safety_subspace.pt
Step 1 complete.
GPU memory cleared.


In [18]:
# Cell 10: Run Step 1 — fine-tune + compute + save subspace
# This cell is independently runnable; it saves checkpoints to disk.

STEP1_DONE_FLAG = MODELS_DIR / "step1_done.flag"

if STEP1_DONE_FLAG.exists():
    print("Step 1 checkpoint found — skipping fine-tuning.")
    print(f"  Safety LoRA  : {SAFETY_LORA_PATH}")
    print(f"  Safety subspace: {SAFETY_SUBSPACE_PATH}")
else:
    # 1a. Load base model and tokenizer
    base_model, tokenizer = load_base_model_and_tokenizer()

    # 1b. Wrap with LoRA
    lora_model = apply_lora(base_model)

    # 1c. Load HH-RLHF data
    chosen_texts, rejected_texts = load_hh_rlhf(
        n_chosen=N_SAFETY_SAMPLES,
        n_rejected=N_BIASED_SAMPLES,
    )
    # Save rejected texts for use in Step 2
    torch.save(rejected_texts, str(DATA_DIR / "rejected_texts.pt"))
    # Save a sample of preference pairs for DPO (Step 3)
    dpo_pairs = list(zip(
        chosen_texts[:N_DPO_PAIRS],
        rejected_texts[:N_DPO_PAIRS],
    ))
    torch.save(dpo_pairs, str(DATA_DIR / "dpo_pairs.pt"))

    # 1d. Safety fine-tune
    trainer = SafetyTrainer(lora_model, tokenizer)
    trainer.fine_tune(chosen_texts)

    # 1e. Compute safety subspace via SVD
    safety_subspace = SafetySubspace(safety_rank=SAFETY_RANK)
    safety_subspace.compute_from_lora(lora_model)
    safety_subspace.save(SAFETY_SUBSPACE_PATH)

    # Mark step done
    STEP1_DONE_FLAG.touch()
    print("Step 1 complete.")

    # Free memory before next step
    del base_model, lora_model
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

Step 1 checkpoint found — skipping fine-tuning.
  Safety LoRA  : /home/jovyan/federated-ethical-llm/models/safety_lora
  Safety subspace: /home/jovyan/federated-ethical-llm/models/safety_subspace.pt
